token 是什么、中文为什么贵

In [33]:
import os
import asyncio
import anthropic
import tiktoken
import httpx
from dotenv import load_dotenv
from zai import ZhipuAiClient
from dataclasses import dataclass

load_dotenv()

@dataclass # class是类级别的共享属性，不是每个实例独立的，使用 @dataclass
class TokenCount:
    text: str = ""
    count: int = 0
    source: str = ""

In [34]:
async def api_token(input: str, client: httpx.AsyncClient) -> TokenCount:
    params = {
        "model":"glm-4.5-air",
        # "temperature": 1,
        "max_tokens": 1,
        "stream_options": {"include_usage": True},
        "stream": False,
        "messages":[
           {
                "role": "user",
                "content": input
           }
        ]
    }
    
    response = await client.post(
        url="https://open.bigmodel.cn/api/paas/v4/chat/completions",
        json=params,
        follow_redirects=True
    )
    response.raise_for_status()   # 加一行:网络/鉴权出错时能立刻看到,而不是后面 KeyError
    res_json = response.json()
    return TokenCount(text=input, count=res_json["usage"]["prompt_tokens"], source="api")

In [35]:
async def tiktoken_calculate(input: str) -> TokenCount:
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = len(enc.encode(input))
    return TokenCount(text=input, count=tokens, source="tiktoken")

In [36]:
async def tokens_counts(inputs: list[str]) -> list[TokenCount]:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.getenv('ZAI_API_KEY')}"
    }
    timeout = httpx.Timeout(connect=30, read=60, write=60, pool=60)

    async with httpx.AsyncClient(headers=headers, timeout=timeout) as client:

        tasks_tik = [asyncio.create_task(tiktoken_calculate(t)) for t in inputs]
        tasks_api = [asyncio.create_task(api_token(input=t, client=client)) for t in inputs]

        tokens: list[TokenCount] = []
        for task in asyncio.as_completed(tasks_tik+tasks_api):
            result = await task
            tokens.append(result)
    
        return tokens

In [38]:
    samples = [
        "Hello, how are you today?",
        "你好,你今天过得怎么样?",
        #"func add(a: Int, b: Int) -> Int { return a + b }",
        #'{"name": "Claude", "age": 2}',
    ]

    tokens: list[TokenCount] = await tokens_counts(inputs=samples)

    for prompt in samples:
        datas = [tc for tc in tokens if tc.text == prompt]
        count_str: str = f"{prompt} 👉🏻 "
        for t in datas:
            count_str = count_str + f"token数:{t.count}(统计来源:{t.source}) "
        print(f"{count_str}")

Hello, how are you today? 👉🏻 token数:7(统计来源:tiktoken) token数:12(统计来源:api) 
你好,你今天过得怎么样? 👉🏻 token数:13(统计来源:tiktoken) token数:12(统计来源:api) 


结果: 
Hello, how are you today? 👉🏻 token数:7(统计来源:tiktoken) token数:12(统计来源:api) 
你好,你今天过得怎么样? 👉🏻 token数:13(统计来源:tiktoken) token数:12(统计来源:api) 
func add(a: Int, b: Int) -> Int { return a + b } 👉🏻 token数:18(统计来源:tiktoken) token数:23(统计来源:api) 
{"name": "Claude", "age": 2} 👉🏻 token数:13(统计来源:tiktoken) token数:18(统计来源:api) 

In [ ]:
为什么不能用 tiktoken 算别家账单？

token并不是我们日常理解的一个单词、一个字，而是模型能够识别、处理、生成的最小文本块

一个token的大小长度不固定，有可能是一句完整的话、短语、单词、单词的部分、符号。

模型只认识数字，无法理解文本，模型接受到一段文本后，会先进行分词（Tokenizer），将一段文本切分成一个个token，给每个token编唯一ID，模型其实是对这些ID做计算和预测

目前绝大多数的分词算法是BPE(Byte Pair Encoding,字节对编码)，核心逻辑是”频率优先“，也就是说在训练过程中，出现频率越高的字符组合，会越容易被打包成一个token

tiktoken 是 OpenAI 预先训练并固定下来的一套分词器(词表 + BPE 合并规则)。它不是"实时用模型数据分词",而是一张已经定好的查找表/规则集。
分词这一步发生在文本进入模型之前,是个确定性的、不需要模型参与的预处理。

每家公司用各自的语料训练出各自的词表和 BPE 合并规则 → 分词器不同 → 同一段文本切出的 token 数和边界都不同 
拿 OpenAI 的分词器去估 GLM/Claude 的用量,必然有偏差。

每 token 单价不同:就算两家碰巧切出一样多的 token,GLM、Claude、GPT 每个 token 收费标准也完全不同,
而且输入 token 和输出 token 通常分开计价(输出往往更贵)。